In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Lab2-Transactions").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
df = spark.read.json("data/transactions_10k.jsonl")
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

print(f"Sukces! Liczba rekordów: {df.count()}")
df.show(5, truncate=False)

Sukces! Liczba rekordów: 10000
+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 5 rows



In [2]:
df.groupBy("category").agg(
    count("tx_id").alias("liczba_tx"),
    round(sum("amount"), 2).alias("suma_PLN"),
    min("amount").alias("min_PLN"),
    max("amount").alias("max_PLN")
).orderBy("category").show()

+-----------+---------+----------+-------+-------+
|   category|liczba_tx|  suma_PLN|min_PLN|max_PLN|
+-----------+---------+----------+-------+-------+
|elektronika|     2542|1520770.69|    9.0| 9999.0|
|    książki|     2574| 851382.08|    5.0|9107.25|
|     odzież|     2453| 849877.55|    5.0|9696.63|
|    żywność|     2431| 789514.43|    5.0|6916.92|
+-----------+---------+----------+-------+-------+



NameError: name '_round' is not defined

In [4]:
df.filter(col("store") == "Gdańsk") \
    .groupBy(window("timestamp", "1 hour")) \
    .agg(round(avg("amount"), 2).alias("avg_amount")) \
    .orderBy("avg_amount").show(1)

df.filter(
    (col("timestamp") >= "2024-03-29 09:00:00") & (col("timestamp") < "2024-03-29 09:30:00")) \
    .groupBy("category").count().show()

df.groupBy(window("timestamp", "15 minutes")) \
    .count() \
    .orderBy(desc("count")).show(1)

+--------------------+----------+
|              window|avg_amount|
+--------------------+----------+
|{2026-04-12 08:00...|    395.01|
+--------------------+----------+
only showing top 1 row

+--------+-----+
|category|count|
+--------+-----+
+--------+-----+

+--------------------+-----+
|              window|count|
+--------------------+-----+
|{2026-04-12 09:15...| 1234|
+--------------------+-----+
only showing top 1 row

